# EDA — PAD-UFES-20

Análisis exploratorio del dataset de lesiones de piel (imágenes clínicas + datos tabulares).
Objetivo: entender el target, el desbalance, los valores faltantes, las variables clínicas y
la estructura por paciente, para fundamentar el split y el modelado posteriores.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from src.data.dataset import load_dataset
from src.data.dataset_utils import CLASSES, MALIGNANT, index_images, integrity_report

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Carga e integridad del dataset

In [ ]:
df = load_dataset()
integrity_report(df, index_images())

In [ ]:
print("filas:", df.shape[0], "| columnas:", df.shape[1])
df.head()

## 2. Variable objetivo

El target es `diagnostic` (6 clases). Miramos su distribución y la vista agrupada
maligno vs benigno (`{BCC, MEL, SCC}` vs el resto).

In [ ]:
counts = df["diagnostic"].value_counts().reindex(CLASSES)
print(counts)

ax = counts.plot(kind="bar", color="steelblue")
ax.set_title("Distribución de diagnósticos (6 clases)")
ax.set_xlabel("diagnóstico"); ax.set_ylabel("n")
for i, v in enumerate(counts):
    ax.text(i, v + 5, str(v), ha="center")
plt.show()

In [ ]:
df["malignant"] = df["diagnostic"].isin(MALIGNANT)
print(df["malignant"].value_counts())
print((df["malignant"].value_counts(normalize=True) * 100).round(1))

## 3. Valores faltantes

Varias columnas clínicas tienen nulos. Los cuantificamos para decidir la estrategia de imputación.

In [ ]:
na = df.isna().mean().sort_values(ascending=False)
na = na[na > 0]
print((na * 100).round(1))

if len(na):
    ax = na.mul(100).plot(kind="barh", color="indianred")
    ax.set_title("% de valores faltantes por columna")
    ax.set_xlabel("% faltante")
    plt.show()

## 4. Variables numéricas

Edad y los dos diámetros de la lesión. Distribución global y por clase.

In [ ]:
num_cols = ["age", "diameter_1", "diameter_2"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df[num_cols].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, num_cols):
    sns.histplot(df[c].dropna(), ax=ax, kde=True, color="steelblue")
    ax.set_title(c)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=df, x="diagnostic", y="age", order=CLASSES, ax=ax)
ax.set_title("Edad por diagnóstico")
plt.show()

## 5. Variables categóricas y de contexto

In [ ]:
cat_cols = ["gender", "region", "fitspatrick", "smoke", "drink",
            "skin_cancer_history", "cancer_history", "pesticide",
            "has_piped_water", "has_sewage_system"]
for c in cat_cols:
    print(f"--- {c} ---")
    print(df[c].value_counts(dropna=False).head(10))
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
df["region"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Región de la lesión")
df["fitspatrick"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Tipo de piel (Fitzpatrick)")
plt.tight_layout(); plt.show()

## 6. Síntomas clínicos de la lesión

Señales booleanas (pica, creció, duele, cambió, sangra, elevación). Vemos su prevalencia
y si se asocian con malignidad.

In [ ]:
symptom_cols = ["itch", "grew", "hurt", "changed", "bleed", "elevation"]
rate = (df.groupby("malignant")[symptom_cols]
          .apply(lambda g: (g == "True").mean()))
print(rate.round(2).T)

df[symptom_cols].apply(lambda s: s.value_counts()).T

## 7. Estructura por paciente (justifica el split agrupado)

Un mismo paciente puede tener varias lesiones e imágenes. Si al hacer el split un paciente
cayera en train y en test a la vez, habría **data leakage**. Por eso el split debe agrupar por
`patient_id`. Acá lo cuantificamos.

In [ ]:
print("pacientes:", df["patient_id"].nunique())
print("lesiones:", df["lesion_id"].nunique())
print("imágenes:", len(df))

per_patient = df.groupby("patient_id").agg(
    n_img=("img_id", "count"),
    n_lesion=("lesion_id", "nunique"),
)
per_patient.describe()

In [ ]:
ax = per_patient["n_img"].value_counts().sort_index().plot(kind="bar", color="steelblue")
ax.set_title("Imágenes por paciente")
ax.set_xlabel("nº de imágenes"); ax.set_ylabel("nº de pacientes")
plt.show()

multi = (per_patient["n_img"] > 1).mean()
print(f"% de pacientes con más de una imagen: {multi*100:.1f}%")

## 8. Proporción de casos con biopsia

`biopsed` indica si el diagnóstico está confirmado por biopsia (ground truth más fuerte).

In [ ]:
print(df["biopsed"].value_counts(dropna=False))
pd.crosstab(df["diagnostic"], df["biopsed"], normalize="index").round(2)

## 9. Imágenes: tamaños

Las fotos son de smartphone, con tamaños variables → habrá que estandarizarlas para la CNN.

In [ ]:
paths = df["img_path"].dropna().sample(min(300, df["img_path"].notna().sum()), random_state=0)
sizes = []
for p in paths:
    with Image.open(p) as im:
        sizes.append(im.size)
sizes = pd.DataFrame(sizes, columns=["width", "height"])
print(sizes.describe())

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(sizes["width"], sizes["height"], alpha=0.3, color="steelblue")
ax.set_xlabel("ancho"); ax.set_ylabel("alto"); ax.set_title("Tamaños de imagen (muestra)")
plt.show()

## 10. Muestra de imágenes por clase

In [ ]:
fig, axes = plt.subplots(1, len(CLASSES), figsize=(18, 3))
for ax, cls in zip(axes, CLASSES):
    row = df[df["diagnostic"] == cls].iloc[0]
    with Image.open(row["img_path"]) as im:
        ax.imshow(im)
    ax.set_title(cls); ax.axis("off")
plt.show()

## 11. Conclusiones

- **Desbalance severo**: MEL (~52) frente a BCC (~845). Reportar macro-F1 / balanced accuracy y
  usar `class_weight` / focal loss; considerar augmentation más agresiva en clases minoritarias.
- **Split agrupado por paciente**: hay pacientes con varias imágenes → obligatorio agrupar por
  `patient_id` para evitar leakage. La estratificación por clase será aproximada por el grupo.
- **Valores faltantes** en varias columnas clínicas → definir imputación en `features/`.
- **Imágenes de tamaño variable** → redimensionar/normalizar en el preprocesamiento de imagen.
- **Señales clínicas** (síntomas, diámetro, edad) parecen aportar información → justifican el
  enfoque multimodal (tabular + imagen).